# Instalación y configuración

## Paso 1: Descargar Docker Desktop

Ir a: [Link de descarga](https://www.docker.com/products/docker-desktop/)

En Windows:

- Ejecutar el instalador
- Aceptar configuración por defecto
- Reiniciar el computador si lo pide

```{note}
**Recordatorio:** Recuerda manetener Docker Desktop abierto mientras estemos trabajando con él

## Paso 2: Crear una carpeta para nuestro proyecto

Es necesario tener una carpeta en donde iremos dejando nuestros archivos para que todo funciones bien y este bien estructurado

## Paso 3: Buscar nuestros datos

En este caso lo haremos con un archivo CSV, recomiendo [esta página](https://datos.gob.cl/), para buscar distintos tipos de datos de interes. En este caso ocuparemos [estos](https://datos.gob.cl/dataset/precios-consumidor1), los cuales son los precios al consumidor de los principales alimentos de la canasta familiar que son capturados en supermercados, ferias libres, carnicerías y panaderías del país. 

```{note}
**Recordatorio:** Dejarlos dentro de nuestra carpeta



## Paso 4: Configurar Logstash

Para esto haremos un pipeline de Logstash el cual tiene siempre 3 partes:

1. input: ¿De dónde vienen los datos?
2. filter: ¿Cómo transformamos esos datos?
3. output: ¿A dónde enviamos esos datos?

Sigue exactamente esta estructura en este caso

![Esquema](imagenes/Logstash.png)

### Sección Input
```
input {
  file {
    path => "/datos_csv/precio_consumidor_publico_2025.csv"
    start_position => "beginning"
    sincedb_path => "/dev/null"
    codec => plain { charset => "UTF-8" }
  }
}
```



In [ ]:
input {
  file {
    path => "/datos_csv/precio_consumidor_publico_2025.csv"
    start_position => "beginning"
    sincedb_path => "/dev/null"
    codec => plain { charset => "UTF-8" }
  }
}

filter {
  csv {
    separator => ","
    quote_char => '"'
    skip_header => true
    columns => [
      "anio","mes","semana","fecha_inicio","fecha_termino","id_region",
      "region","sector","tipo_punto","grupo","producto","unidad",
      "precio_minimo","precio_maximo","precio_promedio"
    ]
  }
  mutate {
    gsub => [
      "precio_promedio", ",", "."
    ]
  }
  mutate {
    convert => {
      "anio" => "integer"
      "mes" => "integer"
      "semana" => "integer"
      "id_region" => "integer"
      "precio_minimo" => "float"
      "precio_maximo" => "float"
      "precio_promedio" => "float"
    }
  }
  date {
    match => [ "fecha_inicio", "yyyy-MM-dd" ]
    target => "@timestamp"
  }
}

output {
  elasticsearch {
    hosts => ["http://elasticsearch:9200"]
    index => "precios_consumidor_2026"
    manage_template => false
  }
}